<a href="https://colab.research.google.com/github/Castlebin/Hands-On-Large-Language-Models-CN/blob/my_master/0_my_code/ch01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Chapter 1 - Introduction to Language Models</h1>
<i>Exploring the exciting field of Language AI</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb)

---

This notebook is for Chapter 1 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [1]:
# %%capture
# !pip install transformers==4.41.2 accelerate==0.31.0

In [2]:
# 要安装的包
!pip install transformers accelerate vllm
# 安装我自己的包
!pip install dl_d2l matplotlib_cn

In [3]:
import os
from dl_d2l.util import colab_util

# 缓存目录
base_data_dir = colab_util.get_base_data_dir()
print(f"base data dir: {base_data_dir} \n")

# 数据集缓存目录
datasets_dir = os.path.join(base_data_dir, "ML", "Datasets")
os.makedirs(datasets_dir, exist_ok=True)
print(f"datasets dir: {datasets_dir} \n")

# 让 matplotlib 绘图 支持中文显示
from matplotlib_cn import matplotlib_util
matplotlib_util.enable_chinese()

# 使用 AutoModelForCausalLM.from_pretrained() 下载模型时，可以通过 cache_dir 指定缓存目录，下面将使用到
# huggingface 缓存目录
hf_cache_dir = os.path.join(base_data_dir, "ML", "huggingface")
print(f"huggingface cache_dir: {hf_cache_dir} \n")

Current environment is Google Colab, mounting Google Drive...
Mounted at /content/drive
Google Drive data directory ready: /content/drive/MyDrive/data
base data dir: /content/drive/MyDrive/data 

datasets dir: /content/drive/MyDrive/data/ML/Datasets 

huggingface cache_dir: /content/drive/MyDrive/data/ML/huggingface 



# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).


使用 transformers 库，加载模型和它的 tokenizer 。这里使用 Microsoft 的 Phi-3

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
## 加载模型
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
    cache_dir=hf_cache_dir    # 指定缓存目录
)
# 加载模型的 tokenizer (Embedding 模型)
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    cache_dir=hf_cache_dir    # 指定缓存目录
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [5]:
from transformers import pipeline

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

Device set to use cuda
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Finally, we create our prompt as a user and give it to the model:

In [6]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

 Why did the chicken join the band? Because it had the drumsticks!


# -----  英文原版内容结束 --------
下面是扩展内容

## 使用国产大模型 Qwen

按同样的方式，来尝试一下国产大模型 Qwen



In [7]:
# 使用 国产大模型千问 Qwen/Qwen2.5-0.5B-Instruct
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

## 1. 加载模型 和 它的 tokenizer
model_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype="auto",
    trust_remote_code=False,
    cache_dir=hf_cache_dir    # 指定缓存目录
)
## 2. 加载 tokenizer
tokenizer_qwen = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=hf_cache_dir    # 指定缓存目录
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
from transformers import pipeline

# Create a pipeline
generator_qwen = pipeline(
    "text-generation",
    model=model_qwen,
    tokenizer=tokenizer_qwen,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [9]:
messages_qwen = [
    {"role": "user", "content": "讲一个关于鸡的笑话"}
]

output_qwen = generator_qwen(messages_qwen)
print(output_qwen[0]["generated_text"])

在一片广阔的田野上，一只小鸡正在悠闲地觅食。突然，它发现前方有一只大公鸡正站在它的面前，准备抢食。小鸡感到非常尴尬和困惑，但它并没有放弃，而是用它那尖锐的小嘴啄了啄大公鸡的头，然后飞快地跑开了。
大公鸡看到小鸡的样子，也感到很惊讶，但它没有生气，反而对小鸡说：“你真是个聪明的小家伙，我从来没有见过这么聪明的鸡！”说完，它就飞快地向后退去，留下了一串清脆的叫声。
这个故事告诉我们，有时候，我们不需要过于在意别人的反应，只要保持自己的冷静和智慧，就能找到解决问题的方法。


# 第一章 -- 认识LLM

In [10]:
# 检查当前机器是可以使用 GPU，并且已经安装了正确的 CUDA 版本
!nvidia-smi

Sun Mar 22 10:24:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P0             27W /   70W |    8395MiB /  15360MiB |     14%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1.1 Qwen/Qwen2.5-0.5B-Instruct
假定用户已经对机器学习（ML） 和深度学习（DL） 有一定的了解， 第一部就是加载一个模型进行推理预测。可以对模型和 tokenizer 进行分别加载.  

使用 huggingface 社区的 transformers 包

In [11]:
import os

# 后面就不用每次都要显式的设置模型下载的目标路径 cache_dir 了
os.environ["HF_HOME"] = hf_cache_dir

In [12]:
!echo $HF_HOME

/content/drive/MyDrive/data/ML/huggingface


In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer


# 加载模型和 tokenizer （第一次加载的时候会进行下载）
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)
# 查看 模型结构是什么？
print(model)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [14]:
# 这里有一个 Special token, 打印看一下是什么token呢？
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

# tokenizer.all_special_tokens
## 这些是模型的特殊 token
tokenizer.special_tokens_map

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'eos_token': '<|im_end|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<|im_start|>',
  '<|im_end|>',
  '<|object_ref_start|>',
  '<|object_ref_end|>',
  '<|box_start|>',
  '<|box_end|>',
  '<|quad_start|>',
  '<|quad_end|>',
  '<|vision_start|>',
  '<|vision_end|>',
  '<|vision_pad|>',
  '<|image_pad|>',
  '<|video_pad|>']}

In [15]:
# 每一个模型的 special token 都是不一样的（但是大同小异）
# tokenizer = AutoTokenizer.from_pretrained("/openbayes/input/input0")
# tokenizer.special_tokens_map

## 1.2 基础使用方式
加载模型之后可以怎么使用呢？可以直接使用原始的 model 和 tokenizer 进行推理；

- step1: 加载模型
- step2: 构建 prompt 和 tokenizer
- step3: 推理和解码

刚刚已经加载过模型了，所以直接进行 step2

In [16]:
prompt = "讲一个猫有关的笑话？"
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

#  想一下输入文本变成什么格式
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

print(text)
print("====" * 10)
print(model_inputs)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
讲一个猫有关的笑话？<|im_end|>
<|im_start|>assistant

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  99526,  46944, 100472,
         101063,   9370, 109959,  11319, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


In [17]:
# step3: 推理和解码
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

猫为什么总是在家里待着？
因为家里有猫。


## 1.3 使用 transformers 的 pipeline 简化流程
以上流程明显很繁琐，我们使用 transformers 的 pipeline 来简化流程

In [18]:
from transformers import pipeline


# step1: 生成 pipeline
# return_full_text=False 表示只返回新生成的文本，不包含输入的prompt
# return_full_text=True 则会返回完整文本，包含输入的prompt和生成的新文本
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,  # 只返回新生成的文本部分
    max_new_tokens=500,
    do_sample=False
)

# step2: 构建 prompt
messages = [
    {"role": "user", "content": "写一个和猫有关的笑话."}
]

# step3，输出并解码
output = generator(messages)
print(output[0]["generated_text"])

Device set to use cuda


好的，以下是一个关于猫的笑话：

有一天，一只猫在森林里迷路了。它四处张望，但什么也看不见。突然，一只小鸟飞过来，对猫说：“你看起来很困惑，是不是遇到了什么困难？”猫回答道：“是啊，我找不到回家的路。”小鸟继续说道：“没关系，我会带你找到回家的路的。但是，你需要告诉我，你在哪里见过这个鸟儿吗？”

猫点了点头，然后向小鸟展示了它的爪子。小鸟惊讶地问：“你怎么知道我的名字？”猫回答道：“因为我叫‘小猫’。”

小鸟笑了起来，对猫说：“谢谢你，小猫。我会记住你的名字的，让你知道我在哪里遇到你。”说完，小鸟就飞走了。

这个笑话通过猫和小鸟之间的互动，展现了猫的智慧和机智，同时也传递了一个关于友谊、信任和理解的主题。


In [19]:
# 看一下 output 的完整样子
output

[{'generated_text': '好的，以下是一个关于猫的笑话：\n\n有一天，一只猫在森林里迷路了。它四处张望，但什么也看不见。突然，一只小鸟飞过来，对猫说：“你看起来很困惑，是不是遇到了什么困难？”猫回答道：“是啊，我找不到回家的路。”小鸟继续说道：“没关系，我会带你找到回家的路的。但是，你需要告诉我，你在哪里见过这个鸟儿吗？”\n\n猫点了点头，然后向小鸟展示了它的爪子。小鸟惊讶地问：“你怎么知道我的名字？”猫回答道：“因为我叫‘小猫’。”\n\n小鸟笑了起来，对猫说：“谢谢你，小猫。我会记住你的名字的，让你知道我在哪里遇到你。”说完，小鸟就飞走了。\n\n这个笑话通过猫和小鸟之间的互动，展现了猫的智慧和机智，同时也传递了一个关于友谊、信任和理解的主题。'}]